## Import Libraries

In [2]:
!pip install tensorflow

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses
from tensorflow.keras.models import Model
from keras.datasets import mnist

   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/350.8 MB 2.8 MB/s eta 0:02:06
   ---------------------------------------- 1.8/350.8 MB 4.4 MB/s eta 0:01:20
   ---------------------------------------- 2.9/350.8 MB 4.8 MB/s eta 0:01:13
    --------------------------------------- 4.5/350.8 MB 5.5 MB/s eta 0:01:04
    --------------------------------------- 4.7/350.8 MB 5.6 MB/s eta 0:01:02
    --------------------------------------- 4.7/350.8 MB 5.6 MB/s eta 0:01:02
    --------------------------------------- 5.0/350.8 MB 3.3 MB/s eta 0:01:45
    --------------------------------------- 5.0/350.8 MB 3.3 MB/s eta 0:01:45
    --------------------------------------- 5.0/350.8 MB 3.3 MB/s eta 0:01:45
    --------------------------------------- 5.0/350.8 MB 3.3 MB/s eta 0:01:45
    --------------------------------------- 5.0/350.8 MB 3.3 MB/s eta 0:01:45


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\priya\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\credentials\\call\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



ModuleNotFoundError: No module named 'tensorflow.python'

## Load MNIST Dataset

In [ ]:
(x_train, _), (x_test, _) = mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

## Define an encoder

In [ ]:
class SimpleAutoencoder(Model):
    def __init__(self, latent_dimensions):
        super(SimpleAutoencoder, self).__init__()
        self.encoder = tf.keras.Sequential([
            layers.Input(shape=(28, 28, 1)),
            layers.Flatten(),
            layers.Dense(latent_dimensions, activation='relu'),
        ])
        
        self.decoder = tf.keras.Sequential([
            layers.Dense(28 * 28, activation='sigmoid'),
            layers.Reshape((28, 28, 1))
        ])
    
    def call(self, input_data):
        encoded = self.encoder(input_data)
        decoded = self.decoder(encoded)
        return decoded

## Compiling & Fitting the Encoder

In [ ]:
latent_dimensions = 64
autoencoder = SimpleAutoencoder(latent_dimensions)
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())

autoencoder.fit(x_train, x_train,
                epochs=10,
                batch_size=256,
                shuffle=True,
                validation_data=(x_test, x_test))

## Visualize the data

In [ ]:
encoded_imgs = autoencoder.encoder(x_test).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

n = 6
plt.figure(figsize=(12, 6))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.title("Original")
    plt.axis('off')

    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    plt.title("Reconstructed")
    plt.axis('off')

plt.show()